##### <center><b style="font-size:1.17em;">Simulations</b></center>
<div style="font-size:6px; line-height:8px;">&nbsp;</div>
<center>
Subthreshold clustered <strong>glutamatergic</strong> input and temporally varying fast <strong>GABA<sub>A</sub></strong> synaptic inputs
</center>


- **2610** – ctrl varies nGLUT inputs
- **2611** – varies nGLUT inputs with fixed GABA tGLUT - tGABA = + 10 ms
- **2612** – varies nGLUT inputs with reduced K
- **2613** – varies nGLUT inputs with fixed GABA tGLUT - tGABA = + 10 ms with reduced K
- **2616** – varies nGLUT inputs with reduced K + 0 Naf to eliminate APs
- **2617** – varies nGLUT inputs with fixed GABA tGLUT - tGABA = + 10 ms with reduced K + 0 Naf to eliminate APs


In [ ]:
# choose sim and cell_type (can be 'dspn' or 'ispn)
sim = '2617'
cell_type = 'ispn' 

In [ ]:
############################################## SETTINGS ##############################################  
# settings
model = 3

# import default settings
import os, sys

cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

os.chdir(main_dir)
sys.path.insert(0, main_dir)

%run -i settings.py

record_3D = True
record_3D = True
record_3D_mechs = True
record_3D_Ca = True
record_Ca = False
record_currents = True
record_branch = True 
record_mechs = True
record_path_dend = True 
record_path_spines = False   
# record_3D_impedance = True  
    
# show overall summary plot and save
showplot = True 

if compare_last_digit(sim, 2) or compare_last_digit(sim, 3) or compare_last_digit(sim, 6) or compare_last_digit(sim, 7):
    if cell_type == 'dspn':
        scale_factors = {'kcnq': 0.5}
    else:
        scale_factors = {'kcnq': 0.5, 'kir': 0.5, 'kaf': 0.5}
    v_init = -85.4087

if compare_last_digit(sim, 6) or compare_last_digit(sim, 7):
    if cell_type == 'dspn':
        scale_factors = {'naf':0, 'kcnq': 0.5}
    else:
        scale_factors = {'naf':0, 'kcnq': 0.5, 'kir': 0.5, 'kaf': 0.5}
    v_init = -85.4087

    
dend_glut = ['dend[15]']
if record_branch:
    dend_glut2 = ['dend[8]', 'dend[10]', 'dend[12]', 'dend[14]']
glut = True

glut_frequency = 1000 # every 1ms
AMPA = True
NMDA = True
g_AMPA = 0.001

# generate rel_glut_onsets
dstep1 = int(1/glut_frequency *1e3)

N_GLUT_dict = {
    'a': 0,     'b': 1,
    'c': 2,     'd': 3,    
    'e': 4,     'f': 5,
    'g': 6,     'h': 7,
    'i': 8,     'j': 9,  
    'k': 10,    'l': 11, 
    'm': 12,    'n': 13, 
    'o': 14,    'p': 15, 
    'q': 16,     
}  

# Phasic gaba
if compare_last_digit(sim, 0) or compare_last_digit(sim, 2) or compare_last_digit(sim, 6):
    gaba = False
else:
    gaba = True

gaba_frequency = 50 
g_GABA = 0.001
gaba_reversal = -60
vary_gaba_time = False # if true timing of gaba is varied relative to glut; if false then vary glut relative to gaba
num_gabas = 12
timing = 160   

sims261x = [f'{prefix}{letter}' for prefix in ['2610', '2611', '2612', '2613', '2616', '2617'] for letter in N_GLUT_dict]
sim_root = ''.join(c for c in sim if c.isdigit())

# Checking if sim is in the combined list
    
record_path_dend

sim_time = 430
save = False
dt =  0.025
deltat = dt * ds

record_path_dend = True

In [ ]:
# select 12 synapses that won't be eliminated when dendrites removed if specify dend2remove
from analysis_functions import *

# dend2remove = ['dend[55]', 'dend[19]', 'dend[42']

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove)
cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])


cell_coordinates = np.array(cell_coordinates, dtype=object)

width=600
height=600
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, color='grey', height=height, width=width)

fig_morphology.show()

In [ ]:
# chose independent synaptic sites; one per dendrite; exclude site of upstate
cell_coordinates_df = pd.DataFrame(cell_coordinates, columns=['section', 'rel.x', 'x', 'y', 'z', 'dist', 'diam'])
dend_df = cell_coordinates_df[cell_coordinates_df.iloc[:, 0] != 'soma[0]']

# target conditions
n = 12
dend_glut = ['dend[15]']

# search for the lowest random_state satisfying both uniqueness and exclusion
for random_state in range(1, 10000):
    sampled_df = dend_df.sample(n=n, replace=False, random_state=random_state)
    dend_gaba_full = sampled_df['section'].tolist()
    
    # check conditions
    unique_ok = len(dend_gaba_full) == len(set(dend_gaba_full))
    exclude_ok = not any(d in dend_gaba_full for d in dend_glut)
    
    if unique_ok and exclude_ok:
        print(f"random_state: {random_state}")
        break

In [ ]:
# choose inputs
cell_coordinates_df = pd.DataFrame(cell_coordinates, columns=['section', 'rel.x', 'x', 'y', 'z', 'dist', 'diam'])
dend_df = cell_coordinates_df[cell_coordinates_df.iloc[:, 0] != 'soma[0]']

# sample N=250 rows without replacement
if cell_type == 'dspn':
    random_state=9
else:
    random_state=71
    
sampled_df = dend_df.sample(n=n, replace=False, random_state=random_state)
sampled_df = sampled_df.sort_index()
indices = sampled_df.index.to_list()
N_GABA = len(indices)

dend_gaba_full = cell_coordinates_df.loc[indices, 'section'].tolist()
gaba_locations_full = cell_coordinates_df.loc[indices, 'rel.x'].tolist()

In [ ]:
soma_v_master = []
dend_v_master = []

# Determine which list 'sim' belongs to
sim_list=None
sim_list = [s for s in sims261x if s.startswith(sim_root)]
    
if sim_list:
    for sim in tqdm(sim_list):
        sim_letter = ''.join(c for c in sim if c.isalpha())
    
        dend_gaba = dend_gaba_full[0:num_gabas]
        gaba_locations = gaba_locations_full[0:num_gabas]
        rel_gaba_onsets = rel_gaba_onset(n=num_gabas, N=num_gabas)

        num_gluts = get_(sim=sim, _dict=N_GLUT_dict)


        glut = True
        if num_gluts == 0:
            glutamate_locations = spine_locator(cell_type=cell_type, specs=specs, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, dend_glut=dend_glut, num_gluts=1, method=method, dend2remove=dend2remove)
            glut = False
        else:
            glutamate_locations = spine_locator(cell_type=cell_type, specs=specs, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, dend_glut=dend_glut, num_gluts=num_gluts, method=method, dend2remove=dend2remove)

        if dstep1 > 0:
            rel_glut_onsets = list(range(0, num_gluts * dstep1, dstep1))
        else:
            rel_glut_onsets = 0 * num_gluts  # num_gluts repetitions of glut_time
    
        record_dendrite = dend_glut[0]
        record_location = [glutamate_locations[0]]

        data_dict = {
            'v_all_3D': {},
            'Ca_all_3D': {},
            'imp_all_3D': {},
            'i_mechs_3D': {},
            'v_dend_tree': {},
            'v_spine_tree': {},
            'v_branch': {},
            'vsoma': [],
            'vdend': [],
            'v_dend_activated': {},
            'vspine': [],
            'v_spine_activated': {},
            'Ca_soma': [],
            'Ca_dend': [],
            'Ca_spine': [],
            'timing': [],
            'dists': [],
            'dists_spine': [],
            'i_mechs_dend': {},
            'i_mechs_dend_tree': {},
            'i_mechs_spine_tree': {},
            'i_ampa': {},
            'i_nmda': {},
            'i_gaba': {},
            'g_gaba': {},
            'record_dists':[],
            'record_spine':[],
            'spine_dist': [],
            'capacitance': [],
            'timestamp': {}
            }

        # get coordinates for all unique segments within all sections
        cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove)

        # scale conductances
        if scale_factors is not None:
            printed = set()  # track which mechanisms have already been reported
            for sec in h.allsec():
                for seg in sec:
                    for mech, sf in scale_factors.items():
                        if hasattr(seg, mech):
                            getattr(seg, mech).sf = sf
                            if mech not in printed:
                                if mech in ['cal12', 'cal13', 'can', 'car', 'cav32', 'cav33']:
                                    print(f"permeability (cm/s) scaled: '{mech}' scaling factor = {sf}")
                                else:
                                    print(f"conductance (S/cm²) scaled: '{mech}' scaling factor = {sf}")
                                printed.add(mech)

 
        mechs=['pas', 'kdr', 'naf', 'kaf', 'kas', 'kir', 'kcnq', 'cal12', 'cal13', 'can', 'car', 'cav32', 'cav33', 'sk', 'bk']
        mech_distr_3D = record_mechs_distr_3D(cell=cell, mechs=mechs)

        # set these to False is they are not already assigned
        glut_reversal = globals().get('glut_reversal', 0) 
        vary_location = globals().get('vary_location', False)  
        dend_glut_range = globals().get('dend_glut_range', None)  
        vary_gaba_location = globals().get('vary_gaba_location', False)   
        
        # rec_all_path = globals().get('rec_all_path', False)  # default is False
        # if False records all v from spine / dendrite to soma
        # if True records at all unique sites including those distal to the synapse

        # determine if axoshaft or axospinous glutamatergic synapse
        axoshaft = globals().get('axoshaft', False)
        axospine = not axoshaft    

        # Common parameters
        common_params = {
            'cell_type': cell_type, 
            'specs': specs, 
            'spine_per_length':spine_per_length,
            'frequency': frequency,
            'd_lambda': d_lambda,
            'glut_reversal': glut_reversal,
            'num_gluts': num_gluts,
            'gaba_reversal': gaba_reversal,
            'num_gabas': num_gabas,
            'sim_time': stim_time,
            'dt': dt,
            'v_init': v_init,
            'dend2remove': dend2remove,
            'neck_dynamics': neck_dynamics,
            'soma_diameter': soma_diameter
        }

        if not vary_location:
            syn_reversals_params = {
                'dend_glut': dend_glut,
                'glutamate_locations': glutamate_locations,
                'dend_gaba': dend_gaba,
                'gaba_locations': gaba_locations,
                **common_params
            }
            glut_reversals, gaba_reversals = syn_reversals(**syn_reversals_params)

        else:
            syn_reversals_params = {
                **common_params
            }
            if vary_gaba_location:
                syn_reversals_params.update({
                    'dend_glut': dend_glut,
                    'glutamate_locations': glut_locations,
                    'dend_gaba': dend_gaba_range,
                    'gaba_locations': gaba_locations_range,
                })
                glut_reversals, gaba_reversals_range = syn_reversals(**syn_reversals_params)
                gaba_reversals_range = gaba_reversals_range * len(gaba_locations_range)
            else:
                syn_reversals_params.update({
                    'dend_glut': dend_glut_range,
                    'glutamate_locations': glut_locations_range,
                    'dend_gaba': dend_gaba,
                    'gaba_locations': gaba_locations,
                })
                glut_reversals_range, gaba_reversals = syn_reversals(**syn_reversals_params)
                glut_reversals_range = glut_reversals_range * len(glut_locations_range)


        
        # time stamp date
        time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

         # set variables for this simulation
        if vary_gaba_time:
            glut_time = stim_time
            gaba_time = timing
        else:
            glut_time = timing
            gaba_time = stim_time
        if vary_location:
            if vary_gaba_location: 
                dend_gaba = [dend_gaba_range[ii]]
                gaba_reversals = [gaba_reversals_range[ii]] if num_gabas == 1 else gaba_reversals_range[ii]

            else:
                dend_glut = [dend_glut_range[ii]]
                glut_reversals = [glut_reversals_range[ii]] if num_gluts == 1 else glut_reversals_range[ii]

            record_dendrite = record_dends[ii]
            glutamate_locations = [glut_locations_range[ii]]
            gaba_locations = [gaba_locations_range[ii]]
            record_location = [record_locs[ii]]

        if tonic:
            gbar_gaba = gbar_gaba_range[ii]
            print('tonic GABA ', gbar_gaba)
        else:
            gbar_gaba = 0

        # label from 1 to N+1 for R analysis
        protocol = 'sim{}'.format(sim_letter)

        metadata = {}
        # Add fixed variables to metadata
        for name in variable_names:
            try:
                if name not in metadata:
                    metadata[name] = eval(name)
            except NameError:
        #         print(f"Variable {name} not found!") # no need to print not found variables
                continue

        # Run sim
        i_recording_site, v_recording_site, v_all_3D, Ca_all_3D, imp_all_3D, i_mechs_3D, vspine, v_spine_activated, vdend, \
        v_dend_activated, vsoma, v_dend_tree, v_spine_tree, Ca_spine, Ca_dend, Ca_soma, \
        i_mechs_dend, i_mechs_dend_tree, i_mechs_spine_tree, v_branch, zdend, ztransfer, \
        ampa_currents, nmda_currents, gaba_currents, gaba_conductances, record_dist, \
        record_spine, spine_dist, cap, dt, burn_time, start_time = \
        msNEURONsim(sim_time = sim_time, 
                 stim_time = stim_time,
                 baseline = baseline,
                 glut = glut, 
                 glut_frequency = glut_frequency, 
                 glutamate_locations = glutamate_locations, 
                 glut_reversals = glut_reversals,
                 glut_time = glut_time, 
                 num_gluts = num_gluts, 
                 dend_glut = dend_glut, 
                 g_AMPA = g_AMPA,
                 ratio = ratio,
                 gaba = gaba, 
                 gaba_frequency = gaba_frequency, 
                 gaba_locations = gaba_locations, 
                 gaba_reversals = gaba_reversals,
                 gaba_time = gaba_time, 
                 g_GABA = g_GABA, 
                 num_gabas = num_gabas, 
                 dend_gaba = dend_gaba, 
                 specs = specs, 
                 scale_factors = scale_factors, 
                 gaba_tau1 = gaba_tau1,
                 gaba_tau2 = gaba_tau2,
                 rel_gaba_onsets = rel_gaba_onsets, 
                 rel_glut_onsets = rel_glut_onsets,
                 frequency = frequency,
                 d_lambda = d_lambda,
                 dend2remove = dend2remove,
                 v_init = v_init,
                 AMPA = AMPA,
                 NMDA = NMDA,
                 physiological = physiological,
                 timing_range = timing_range,
                 add_noise = add_noise,
                 beta = beta,                   
                 B = B,                      
                 add_sine = add_sine, 
                 axoshaft = axoshaft,
                 cell_type = cell_type,
                 current_step = current_step,
                 step_current = step_current,
                 step_duration = step_duration,
                 step_start = step_start,
                 holding_current = holding_current,
                 add_ramp = add_ramp,
                 ramp_amplitude = ramp_amplitude,
                 Cm = Cm,
                 Ra = Ra,
                 spine_per_length = spine_per_length,
                 spine_neck_diam = spine_neck_diam,
                 spine_neck_len = spine_neck_len,
                 spine_head_diam = spine_head_diam,
                 soma_diameter = soma_diameter,
                 neck_dynamics = neck_dynamics,
                 space_clamp = space_clamp,
                 record_dendrite = record_dendrite, 
                 record_location = record_location, 
                 record_currents = record_currents,
                 record_branch = record_branch,
                 dend_glut2 = dend_glut2, 
                 record_mechs = record_mechs,
                 mechs2record = mechs2record,
                 record_path_dend = record_path_dend,   
                 record_path_spines = record_path_spines,  
                 record_all_path = record_all_path,
                 record_3D = record_3D,         
                 record_3D_impedance = record_3D_impedance,
                 freq = freq,                   
                 record_3D_mechs = record_3D_mechs,    
                 record_Ca = record_Ca,
                 record_3D_Ca = record_3D_Ca,
                 tonic = tonic,               
                 gbar_gaba = gbar_gaba,
                 rectification = rectification,       
                 distributed = distributed,         
                 gaba_params = gaba_params,
                 tonic_gaba_reversal = tonic_gaba_reversal,
                 dt = dt,
                 ds_imp = ds_imp,
                 voltage_clamp = voltage_clamp,
                 holding_potential = holding_potential,
                 voltage_clamp_site = voltage_clamp_site,
                 voltage_clamp_spine = voltage_clamp_spine,
                 voltage_clamp_loc = voltage_clamp_loc,
                 Rs = Rs,
                 downsample=downsample,
                 ds=ds
        )


        ind1 = 1 
        ind2 = int(sim_time/deltat)
        t2 = np.arange(1, int(sim_time/deltat), 1) * deltat - deltat 
        soma_v_master.append(go.Scatter(x=t2,  y=extract2(vsoma)[ind1:ind2]))
        dend_v_master.append(go.Scatter(x=t2, y=extract2(vdend)[ind1:ind2]))

        # only do if want to view each sim or save sim graphs
        if record_path_dend:
            plots_return(v_tree=v_dend_tree['v'], vspine=vspine, dists=v_dend_tree['dists'], spine_dist=spine_dist, num_gluts=num_gluts, start_time=start_time, burn_time=burn_time, dt=deltat, xaxis_range=[0,300], Nsim_plot=True, Nsim_save=False, sim_image_path=None, time=time)

        update_data_dictionary(data_dict=data_dict, protocol=protocol, v_all_3D=v_all_3D, 
                Ca_all_3D=Ca_all_3D, imp_all_3D=imp_all_3D, i_mechs_3D=i_mechs_3D, 
                vspine=vspine, v_spine_activated=v_spine_activated, vdend=vdend, v_dend_activated=v_dend_activated, 
                vsoma=vsoma, v_dend_tree=v_dend_tree, v_spine_tree=v_spine_tree, Ca_spine=Ca_spine, 
                Ca_dend=Ca_dend, Ca_soma=Ca_soma, timing=timing, i_mechs_dend=i_mechs_dend, 
                i_mechs_dend_tree=i_mechs_dend_tree, i_mechs_spine_tree=i_mechs_spine_tree, v_branch=v_branch, 
                ampa_currents=ampa_currents, nmda_currents=nmda_currents, gaba_currents=gaba_currents, 
                gaba_conductances=gaba_conductances, record_dist=record_dist, time=time, 
                record_currents=record_currents, record_spine=record_spine, spine_dist=spine_dist, cap=cap)

        data_dict['metadata'] = metadata
        data_dict['mechanisms_3D_distribution'] = mech_distr_3D

        # save
        if save:
            simulations_path = 'downsample/{}/model{}/{}/simulations/sim{}/{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim_root, sim_letter)
            os.makedirs(simulations_path, exist_ok=True)
            names = list(data_dict.keys())
            for name in names:
                with open('{}/{}.pickle'.format(simulations_path, name), 'wb') as handle:
                    pickle.dump(data_dict[name], handle, protocol=pickle.HIGHEST_PROTOCOL)  


In [ ]:
fig_soma_master, fig_dend_master = plot3(somaV=soma_v_master, dendV=dend_v_master, yaxis='V (mV)', stim_time=stim_time,
                                         sim_time=sim_time, black_trace=None, gray_trace=0, offset=True, offset_ms=20, x_err_bar=25, baseline=20, 
                                         ds=1, dt=deltat)    
fig_soma_master.show(config={"toImageButtonOptions": {"format": "svg"}})
fig_dend_master.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    sim_image_path = 'downsample/{}/model{}/{}/images/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim_root)
    os.makedirs(sim_image_path, exist_ok=True)
    fig_soma_master.write_image(f"{sim_image_path}/fig_soma_master.svg", format='svg', scale=3)
    fig_dend_master.write_image(f"{sim_image_path}/fig_dend_master.svg", format='svg', scale=3)
    
    

In [ ]:
sim_image_path 
